## Introduction

This notebook is a debugging notebook for the low level Jupyter Comms communications within [CubeVis](https://github.com/casangi/cubevis). The motivation for the ``cubevis`` shift from using only [websockets](https://pypi.org/project/websockets/) to supporting ``websockets``, ``Jupyter Comms`` and ``Colab Comms`` was the desire to support Colab.

Initially, it seemed as though the transition would be relatively straight forward for the original ``websockets`` based implementation. Colab has integrated proxy support:
```
from google.colab.output import eval_js
port = 8888
proxy_url = eval_js(f"google.colab.kernel.proxyPort({port})")
```
However, Google's proxy server does not support full protocol upgrades. In particular, it is not possible to upgrade a proxies port to support the websocket protocol. This truth only came to light after significant effort. After the necessary stages of grief, the decision was made to rework ``cubevis`` communications to support ``websockets``, ``Jupyter Comms`` and ``Colab Comms``, and because this would be a significant investment, we decided to also introduce multiplexed communications for better communication support, both for current commuications as well as future remote execution.

First ``websockets`` support was restored. This proved to be relatively straight forward. The only complication was the introduction of multiplexed communications to replace the original use of a dedicated websocket for each type of communication. However, this allow for development and testing of a code organization which allowed for the specifics of the channels, whether comms based or websockets based, to be isolated behind an abstract base class.

Next support for ``Jupyter Comms`` was introduced. This worksheet served as the development and debugging platform for this flavor of low level communications channel. We started with the most recent version of Jupyter Lab (March 2026) which was first introduced on May 15, 2023. This is probably the most secure version of Jupyter notebooks currently. This led to difficulties.

## What didn't work

### JavaScript access to global variables
The various versions of Classic notebooks have provided access to communications channel via global variables. We checked them all: ``window.jupyterapp``, ``window.Jupyter``, ``window.IPython``, ``window.kernel``. All undefined.

### Lumino symbol walk
Jupyter Lab attaches Lumino to DOM elements by symbol-keyed properties. We walked up ``.jp-NotebookPanel`` ancestors and checked all symbol properties looking for ``sessionContext.session.kernel.createComm``. None were found.

### Broad DOM sweep
Searched all elements of the ``document`` with direct inspection of ``.jp-NotebookPanel`` no comm creation functions were found.

### Access to Jupyter manager
``window._JUPYTERLAB['@jupyter-widgets/jupyterlab-manager']`` was found. It contained ``get`` and ``init`` methods, and we were able to create a manager. However, without access to the manager object which was being used by Jupyter Lab, this was of no use.

### Looking for the manager in use
After loading the widget manager module, tried to access the static instance registry through ``mod.WidgetManager._managers``. However, there were no static properties at all. The _managers registry is a closure variable (``pe.widgetManagerProperty``), a Lumino ``AttachedProperty`` stored as a WeakMap inside the module closure. Not iterable, not accessible.

### Searching plugin descriptors
The module exposes 4 JupyterLab plugin descriptors. We inspected their activate functions for closure state and live service references, but the activate source revealed ``pe.widgetManagerProperty.get(kernelConnection)`` which confirmed that the kernel is in a closure, but the closure variables pe and ee are minified and inaccessible.

### Interception of calls
In the module that we found, we tried to replace ``mod.registerWidgetManager`` function with a spy wrapper to intercept the next panel activation. This failed because the module is read only and cannot set property. It only has a getter function so cannot force redefinition of the function.

### Inspection of custom widgets
We created a real ``ipywidgets.IntSlider``, waited for it to render, then searched for ``_view``, ``_model``, or manager references on the widget's DOM elements. This failed because the widget renders as plain CSS/HTML with no JS object references attached to any DOM node. ``jupyter-widget-view`` custom element is not defined in this build.

### webpack internals
Searched window for Webpack 5 internals, ``__webpack_share_scopes__`` and ``__webpack_require__``,  details that would give access to the shared module scope containing ``@jupyterlab/services``. Both completely sealed. Not exposed on window.

### REST access
We tried to access the information from the REST AIP that Jupyter supports. The ``window.PageConfig`` + ``/api/sessions`` REST fetch fell back to the Jupyter REST API to discover the running kernel ID. This worked and provided a potential path.

With this success, we were able to use Python's ``create_comm().open()``. Instead of JS sending ``comm_open``, Python initiates via the standalone ``comm`` package. Python's comm_open goes over iopub, which JupyterLab delivers to the frontend without target validation. The JavaScript shim received comm_open reliably and the connection was established.

However, the JavaScript acknowledgement via signed a signed ``comm_msg`` over a raw WebSocket failed. JavaScript sent a ``comm_msg`` acknowledgment back over our raw WebSocket but it was never received. The Jupyter server's WebSocket endpoint is a ZMQ↔WebSocket bridge, but only translates messages from its own frontend connection. Our second raw WebSocket sends JSON but ipykernel expects ZMQ multi-frame format. Messages are silently dropped.

### jupyter_bokeh
``jupyter_bokeh`` is supposed to be the package that provides a path for Jupyter integration. The intention was to use ``interactive_communication``, but with Jupyter Lab 4 the frontend extension often fails to attach this object to the model's document at the right time, leading to undefined errors.

### ipywidgets
We tried using ``ipykernel.comm.Comm`` for access to the Jypyter Comms library. The Python side of ``ipywidgets`` does work correctly, but getting a functional, authenticated Comm object in the JavaScript side within a cell is nearly impossible without a manager.

### ipywidgets_bokeh
After achieving bidirectional communications with ``anywidget``, we attempted to use ``ipywidgets_bokeh`` to allow placing the gateway ``anywidget`` within the Bokeh GUI by the user. This would have allowed the debug messages which are displayed in the widget that provides the communications link. Typically, this widget is invisible but when ``os.environ['CUBEVIS_DEBUG']`` is set, then communications debugging information is displayed in the link widget. ``ipywidgets_bokeh`` should have allowed this, but it turned out to be somewhat out of date (using the deprecated ``requirejs``) and also required additional Bokeh registration of ``IPyWidget``.

## Debugging setup with github branch/tag selection

In [ ]:
!pip install casatasks bokeh==3.9 scipy regions

In [1]:
import os
tag = 'jupyter-debug-0170'
os.environ['CUBEVIS_JS_TAG'] = tag
os.environ['CUBEVIS_DEBUG'] = '1'
os.makedirs(os.path.expanduser('~/.casa/data'),exist_ok=True)

This next cell is only required if ``cubevis`` is not already installed **and** we are installing from github. This is especially useful for configuring a *new* runtime in Colab.

In [ ]:
!pip install git+https://github.com/casangi/cubevis.git@{tag}

## Import CommsTransport

In [ ]:
import asyncio
from bokeh.io import show, output_notebook
from cubevis.bokeh.transport._low_level_transport import CommsTransport

## Test Jupyter Comms communications
This initializes the test and sets up a Python message handler for messages from JavaScript to Python.

In [ ]:
output_notebook()

import ipywidgets as widgets
from IPython.display import display

transport = CommsTransport(comm_mgr_id="isolation_test_pipe")

log_box = widgets.Textarea(
    value='',
    layout=widgets.Layout(width='100%', height='200px', font_family='monospace')
)
display(log_box)

def py_receiver(msg):
    # Update of all ipywidgets, anywidgets, etc seems to fail when
    # the update happens on a thread other than the main thread... ¯\_(ツ)_/¯
    from pathlib import Path
    with open(Path.home() / "debug.txt", "a") as f:
        f.write(f"py_receiver: {msg}\n")
    log_box.value += f"✅ Python Received: {msg}\n"

transport.set_message_callback(py_receiver)

### Check connection

In [ ]:
await transport.connect()
if transport.is_connected():
    print("✅ Connection verified!")
else:
    # Give it another 2 seconds if you just ran Cell 1
    import asyncio
    await asyncio.sleep(2)
    print(f"Final Connection Status: {transport.is_connected()}")

### JavaScipt → Python
This sets up a message handler in JavaScript and also sends a message from JavaScript to Python

In [ ]:
%%javascript
(async function verify_bidirectional() {
    const pipe_id = "isolation_test_pipe";

    function appendOutput(html) {
        const div = document.createElement("div");
        div.innerHTML = html;
        document.body.appendChild(div);
    }

    const isColab = typeof google !== "undefined" && google?.colab?.kernel?.comms;

    if (isColab) {
        const bc_rx = new BroadcastChannel(`cubevis_rx_${pipe_id}`)
        const bc_tx = new BroadcastChannel(`cubevis_tx_${pipe_id}`)

        bc_rx.onmessage = (event) => {
            const envelope = event.data
            console.log("📢 JS RECEIVED FROM PYTHON:", envelope)
            appendOutput(`<div style="padding:10px;background:#eef;border:1px solid #2196f3">✅ Received: ${JSON.stringify(envelope)}</div>`)

            // Send back j2p ack so Python's await send_message() can complete.
            // _comm_mgr waits for a response with matching request_id and direction='j2p'.
            try {
                const parsed = JSON.parse(envelope.data)
                const entries = Object.fromEntries(parsed.entries)
                const request_id = entries.request_id
                if (request_id) {
                    bc_tx.postMessage({
                        type: "cubevis_message",
                        comm_mgr_id: pipe_id,
                        data: JSON.stringify({ type: "map", entries: [
                            ["comm_id",     entries.comm_id || ""],
                            ["message_id",  entries.message_id || ""],
                            ["message",     {}],
                            ["direction",   "j2p"],
                            ["request_id",  request_id]
                        ]})
                    })
                    console.log("📤 Sent j2p ack for request_id:", request_id)
                }
            } catch(e) {
                console.log("Could not send ack:", e)
            }
        }

        // JS->Python still via comms.open
        const channel = await google.colab.kernel.comms.open(pipe_id, {})
        channel.send({ text: "Checking JS -> PY path..." });
        appendOutput("📡 JS -> PY Message Sent. Now run the Python 'You win' cell.");

    } else {
        const comm = window["cubevis_" + pipe_id]?.comm;
        if (!comm) {
            element.append("❌ Bridge not found. Ensure the setup cell ran and the widget is blue.");
            return;
        }

        comm.onMsg = (msg) => {
            const data = msg.content.data;
            console.log("📢 JS RECEIVED FROM PYTHON:", data);
            const div = document.createElement("div");
            div.style.cssText = "padding:10px;margin-top:10px;background:#eef;border:1px solid #2196f3";
            div.innerHTML = `✅ <b>Success!</b> Received: ${JSON.stringify(data)}`;
            element.append(div);
        };

        comm.send({ text: "Checking JS -> PY path..." });
        element.append("📡 JS -> PY Message Sent. Now run the Python 'You win' cell.");
    }
})();

### Python → JavaScript

In [ ]:
await transport.send_message({"type": "FINISH_TEST", "content": "You win!"})